# 스케치 변환 미리보기

utils.py의 `convert_to_sketch_query()`와 동일한 파라미터:
- GaussianBlur(5, 5, sigma=1.0)
- Canny(30, 120)
- dilate(2×2, 1회)
- 흰 배경 + 검은 윤곽선

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
def convert_to_sketch(image: Image.Image) -> Image.Image:
    """utils.py의 convert_to_sketch_query()와 동일한 로직"""
    img_array = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    blurred   = cv2.GaussianBlur(img_array, (5, 5), 1.0)
    edges     = cv2.Canny(blurred, 30, 120)
    edges     = cv2.dilate(edges, np.ones((2, 2), np.uint8), iterations=1)
    sketch    = 255 - edges
    sketch_rgb = cv2.cvtColor(sketch, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(sketch_rgb)


def convert_to_sketch_clean(image: Image.Image, min_area: int = 500) -> Image.Image:
    """
    배경 노이즈 제거 버전.
    Canny 후 윤곽선 면적이 min_area 미만인 작은 조각(노이즈)을 제거하고
    큰 윤곽선만 흰 배경에 그립니다.
    
    min_area: 이 값보다 작은 윤곽선은 노이즈로 판단해 제거 (기본 500px²)
              값이 클수록 더 많이 제거됨
    """
    img_array = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    blurred   = cv2.GaussianBlur(img_array, (5, 5), 1.0)
    edges     = cv2.Canny(blurred, 30, 120)
    edges     = cv2.dilate(edges, np.ones((2, 2), np.uint8), iterations=1)

    # 윤곽선 추출 후 면적 기준 필터링
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    canvas = np.ones_like(edges) * 255  # 흰 배경
    for cnt in contours:
        if cv2.contourArea(cnt) >= min_area:
            cv2.drawContours(canvas, [cnt], -1, 0, 1)  # 검은 선

    sketch_rgb = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(sketch_rgb)

In [ ]:
# 이미지 경로 입력
IMAGE_PATH = "test.jpg"   # ← 여기에 이미지 경로 입력

original      = Image.open(IMAGE_PATH).convert('RGB')
sketch_orig   = convert_to_sketch(original)
sketch_clean  = convert_to_sketch_clean(original, min_area=500)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(original);     axes[0].set_title('원본 이미지', fontsize=14);        axes[0].axis('off')
axes[1].imshow(sketch_orig);  axes[1].set_title('기존 스케치 (노이즈 있음)', fontsize=14); axes[1].axis('off')
axes[2].imshow(sketch_clean); axes[2].set_title('노이즈 제거 스케치', fontsize=14); axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# min_area 값별 비교 (튜닝용)
# 값이 클수록 작은 조각을 더 공격적으로 제거
areas = [100, 500, 1000, 3000]

fig, axes = plt.subplots(1, len(areas), figsize=(16, 5))
for ax, area in zip(axes, areas):
    result = convert_to_sketch_clean(original, min_area=area)
    ax.imshow(result)
    ax.set_title(f'min_area={area}', fontsize=12)
    ax.axis('off')

plt.suptitle('min_area 값별 노이즈 제거 비교', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Canny 임계값 비교 (파라미터 튜닝용)
configs = [
    (10,  60,  "low (10/60)"),
    (30, 120,  "default (30/120)"),
    (50, 150,  "high (50/150)"),
    (80, 200,  "very high (80/200)"),
]

img_gray = cv2.cvtColor(np.array(original), cv2.COLOR_RGB2GRAY)
blurred  = cv2.GaussianBlur(img_gray, (5, 5), 1.0)

fig, axes = plt.subplots(1, len(configs), figsize=(16, 5))
for ax, (lo, hi, label) in zip(axes, configs):
    edges  = cv2.Canny(blurred, lo, hi)
    edges  = cv2.dilate(edges, np.ones((2, 2), np.uint8), iterations=1)
    sketch = 255 - edges
    ax.imshow(sketch, cmap='gray')
    ax.set_title(label, fontsize=11)
    ax.axis('off')

plt.suptitle('Canny 임계값별 비교', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 최종 결과 저장
OUTPUT_PATH = "sketch_clean_output.jpg"
sketch_clean.save(OUTPUT_PATH)
print(f"저장 완료: {OUTPUT_PATH}")